# 03 — Baseline Models Evaluation & The Clean Benchmark Reality
**Course:** BCSE209P — Machine Learning Lab  
**Students:** P Naga Venkata Kushal (23BDS0133) & Koteru Tarun (23BDS0103)  
**Faculty Guide:** Dr. Dega Nagaraju  
**Repository:** [Reliability-Aware Anomaly Detection](https://github.com/nvk321/Reliability-Aware-Anomaly-Detection)

---
### Objectives
1. Inspect the audited performance of Baseline A (Isolation Forest), Baseline B (ST-AE), and Baseline C (RSTNetNoSRM).
2. Explain why Isolation Forest achieves high performance on the clean SWaT benchmark ($F_1 = 0.7182, \text{FPR} = 0.92\%$).
3. Explain why deep reconstruction architectures without reliability weighting suffer from residual elevation ($F_1 \approx 0.217, \text{FPR} > 99\%$).
4. Establish why Baseline C (RSTNetNoSRM) is the mathematically exact ablation model for R-STNet.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.visualization.plots import apply_scientific_style
apply_scientific_style()

print('Imports successful!')


### 1. Load Audited Clean Benchmark Results
The frozen research results are stored in `results/metrics/`.


In [ ]:
metrics_dir = os.path.join('..', 'results', 'metrics')

files = {
    'Isolation Forest': 'baseline_isolation_forest_swat.json',
    'ST-AE': 'baseline_stae_swat.json',
    'RSTNetNoSRM (Baseline C)': 'baseline_rstnet_no_srm_swat.json',
    'R-STNet (Proposed)': 'proposed_rstnet_swat.json',
}

benchmark_records = []
for model_name, filename in files.items():
    filepath = os.path.join(metrics_dir, filename)
    with open(filepath, 'r') as f:
        data = json.load(f)
    raw = data['raw_pointwise_metrics']
    benchmark_records.append({
        'Model': model_name,
        'F1': raw['f1'],
        'Precision': raw['precision'],
        'Recall': raw['recall'],
        'FPR': raw['fpr'],
        'PR-AUC': raw['pr_auc'],
        'ROC-AUC': raw['roc_auc'],
    })

df_bench = pd.DataFrame(benchmark_records)
df_bench


### 2. Visualize Benchmark Comparison
We plot F1 Score, Precision, Recall, and False Alarm Rate (FPR) side-by-side.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

models = df_bench['Model']
x = np.arange(len(models))
width = 0.25

# Left plot: F1, Precision, Recall
ax1.bar(x - width, df_bench['F1'], width, label='F1 Score', color='#1D4ED8')
ax1.bar(x, df_bench['Precision'], width, label='Precision', color='#059669')
ax1.bar(x + width, df_bench['Recall'], width, label='Recall', color='#D97706')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=25, ha='right')
ax1.set_ylabel('Metric Value')
ax1.set_title('Detection Performance on Clean SWaT Benchmark', fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 1.1)

# Right plot: FPR (False Positive Rate)
colors = ['#10B981' if fpr < 0.10 else '#EF4444' for fpr in df_bench['FPR']]
bars = ax2.bar(models, df_bench['FPR'] * 100, color=colors, width=0.5)
ax2.set_xticklabels(models, rotation=25, ha='right')
ax2.set_ylabel('False Positive Rate (FPR %)')
ax2.set_title('False Alarm Rate on Clean SWaT Benchmark', fontweight='bold')

# Label percentages on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 1.5, f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_ylim(0, 115)
plt.tight_layout()
plt.show()


### Viva Voce Key Takeaways — Section C & D: Baseline & Deep Baselines
1. **Isolation Forest Reality:** Isolation Forest achieves the highest clean benchmark performance ($F_1 = 0.7182, \text{FPR} = 0.92\%$). As an axis-aligned tree ensemble, it evaluates sensors independently without joint spatiotemporal coupling. Unrecorded test distribution shifts (e.g., `AIT201` conductivity drift) do not distort its global decision space.
2. **Why ST-AE Fails:** Spatio-temporal autoencoders reconstruct all 51 channels through a dense latent bottleneck. Unrecorded process shifts cause persistent non-zero reconstruction errors across the plant, resulting in near-100% false alarm saturation under clean validation-tuned thresholds.
3. **The Role of RSTNetNoSRM (Baseline C):** Baseline C possesses the exact same 1D TCN + Graph Attention backbone (23,233 parameters) as R-STNet, but sums residuals uniformly without reliability weights. This isolates the Sensor Reliability Module as the sole variable.
